# Clinical Trial Designer â€” GRPO Training Notebook

**OpenEnv Hackathon | Meta PyTorch Ã— Scaler School of Technology**

This notebook trains an LLM to design clinical trials using GRPO (Group Relative Policy Optimization) from HuggingFace TRL + Unsloth for efficient LoRA fine-tuning.

**Platform:** Google Colab (T4 GPU, 16 GB VRAM â€” free tier works with 4-bit quantization)

**What this notebook does:**
1. Installs dependencies (TRL, Unsloth)
2. Connects to the Clinical Trial Designer environment (HF Space)
3. Configures GRPO with LoRA
4. Trains the agent with reward feedback from the environment
5. Evaluates trained vs base model
6. Saves checkpoint to HuggingFace Hub

> **Note:** For dry-run validation (no GPU needed), set `DRY_RUN = True` in Cell 5.

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth for fast LoRA training (2x speed, 50% less VRAM)
!pip install -q unsloth
# Install TRL for GRPO trainer
!pip install -q "trl>=0.29.0"
# Install other dependencies used in this notebook
!pip install -q requests scipy numpy datasets matplotlib huggingface_hub

## 2. Configuration and Environment Connection

The environment runs on our HuggingFace Space. Change `ENV_URL` if running locally.

For stable GRPO rewards, this notebook sends `curriculum_tier` and `freeze_curriculum` on every reset so all completions are scored under a fixed difficulty tier.


In [ ]:
import requests
import json
import os
import random
import re
import ast
import csv
from datetime import datetime, timezone

# === CONFIGURATION ===
ENV_URL = "https://roopalgn-openenv-clinical-trial.hf.space"
DRY_RUN = False
MODEL_SIZE = "1.5b"      # "1.5b", "3b", or "7b"
EPISODES = 50
NUM_GENERATIONS = 6
SEED = 42
ARTIFACT_DIR = "outputs/grpo"
EVAL_EPISODES = 6
EVAL_MAX_NEW_TOKENS = 512
MAX_COMPLETION_LENGTH = 1024
GENERATION_TEMPERATURE = 0.9

# Keep reward evaluation stationary across GRPO updates.
# Tier 3 creates stronger discrimination than warmup-tier episodes.
TRAIN_CURRICULUM_TIER = 3
FREEZE_CURRICULUM = True

# Model size presets (match train.py intent)
MODEL_PRESETS = {
    "1.5b": {"model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "lora_r": 8, "seq_len": 2048, "grad_accum": 4},
    "3b":   {"model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 3072, "grad_accum": 4},
    "7b":   {"model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   "lora_r": 32, "seq_len": 4096, "grad_accum": 8},
}

preset = MODEL_PRESETS[MODEL_SIZE]
print(f"Config: model={MODEL_SIZE}, episodes={EPISODES}, dry_run={DRY_RUN}")
print(f"Preset: {preset}")
print(f"Reward env: tier={TRAIN_CURRICULUM_TIER}, freeze_curriculum={FREEZE_CURRICULUM}")


def env_reset(seed=None, curriculum_tier=TRAIN_CURRICULUM_TIER, freeze_curriculum=FREEZE_CURRICULUM):
    payload = {}
    if seed is not None:
        payload["seed"] = seed
    if curriculum_tier is not None:
        payload["curriculum_tier"] = int(curriculum_tier)
    payload["freeze_curriculum"] = bool(freeze_curriculum)

    resp = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


def env_step(action_type, parameters=None, justification="", confidence=0.5):
    payload = {
        "action_type": action_type,
        "parameters": parameters or {},
        "justification": justification,
        "confidence": confidence,
    }
    resp = requests.post(f"{ENV_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()


def env_ping():
    return requests.get(f"{ENV_URL}/ping", timeout=10).json()


def extract_total_reward(step_result):
    reward = step_result.get("reward", 0.0) if isinstance(step_result, dict) else 0.0
    if isinstance(reward, dict):
        return float(sum(float(v) for v in reward.values()))
    return float(reward)


try:
    print("Ping:", env_ping())
    obs = env_reset(seed=SEED)
    print("Reset OK. Scenario:", obs.get("scenario_description", "")[:100])
    print("Available actions:", obs.get("available_actions", []))
except Exception as e:
    print(f"ERROR: Cannot connect to environment at {ENV_URL}: {e}")
    print("Check that the HF Space is running.")



Config: model=1.5b, episodes=20, dry_run=True
Preset: {'model_name': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit', 'lora_r': 8, 'seq_len': 2048, 'grad_accum': 4}
Ping: {'status': 'ok'}
Reset OK. Scenario: Warmup scenario: EGFR+ solid-tumour chemotherapy with an inflated effect size to
Available actions: ['observe_safety_signal', 'set_primary_endpoint']


## 3. Dry-Run Validation (No GPU Needed)

Run 2 episodes with a random policy to validate the full pipeline: env connection, action, reward, CSV logging. Skip this section if doing real training.

In [2]:
def run_dry_run(n_episodes=5, base_seed=42):
    """Validate reward discrimination: valid action should beat invalid action."""
    print("=== DRY RUN: reward discrimination check ===")
    deltas = []

    for ep in range(n_episodes):
        seed = base_seed + ep

        obs = env_reset(seed=seed)
        available = obs.get("available_actions", ["set_primary_endpoint"])
        valid_action = available[0]
        valid_result = env_step(valid_action)
        valid_reward = extract_total_reward(valid_result)

        # Reset to same seed for fair comparison
        env_reset(seed=seed)
        invalid_result = env_step("synthesize_conclusion")
        invalid_reward = extract_total_reward(invalid_result)

        delta = valid_reward - invalid_reward
        deltas.append(delta)
        print(f"  ep={ep+1:02d} seed={seed} | valid={valid_reward:+.3f} invalid={invalid_reward:+.3f} delta={delta:+.3f}")

    mean_delta = sum(deltas) / len(deltas)
    print(f"\nMean(valid-invalid): {mean_delta:+.3f}")
    if mean_delta > 0.5:
        print("GOOD: reward signal is discriminative and suitable for GRPO.")
    else:
        print("WARNING: weak reward signal. Check server/reward config before long training.")
    return deltas


if DRY_RUN:
    run_dry_run(n_episodes=5, base_seed=SEED)

=== DRY RUN MODE ===
  Episode 1/2 | reward=151.9280 | steps=50 | timeout
  Episode 2/2 | reward=152.3283 | steps=50 | timeout

Dry-run complete. CSV: outputs/grpo/reward_log.csv
Mean reward: 152.1281
=== Pipeline validated. Set DRY_RUN=False for real training. ===


## 4. Load Model with Unsloth

4-bit quantization to fit on Colab T4 (16 GB VRAM). Model size is set by `MODEL_SIZE` in Cell 5.

In [ ]:
if not DRY_RUN:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=preset["model_name"],
        max_seq_length=preset["seq_len"],
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=preset["lora_r"],
        lora_alpha=preset["lora_r"] * 2,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = None

    print(f"Model loaded: {preset['model_name']}")
    model.print_trainable_parameters()
else:
    print("DRY_RUN=True â€” skipping model load.")

## 4b. Define Reward Function (Full Episode Plan)

This section now matches `docs/reward_spec.md` more closely:
- the model must emit a complete ordered action plan,
- the environment executes exactly those actions,
- parse failure gives `-3.0`,
- invalid action sequences give `-3.0`,
- incomplete plans get an additional penalty,
- there is no fixed prerequisite-correct sequence hidden inside the reward function.


In [ ]:
PLAN_PARSE_FAILURE_REWARD = -3.0
INVALID_SEQUENCE_REWARD = -3.0
INCOMPLETE_PLAN_PENALTY = -3.0
MAX_INCOMPLETE_PROGRESS_BONUS = 2.4
PLAN_MAX_ACTIONS = 12
DEFAULT_SAMPLE_SIZE = 240

ALL_ACTION_TYPES = {
    "set_primary_endpoint",
    "set_sample_size",
    "set_inclusion_criteria",
    "set_exclusion_criteria",
    "set_dosing_schedule",
    "set_control_arm",
    "set_randomization_ratio",
    "set_blinding",
    "run_dose_escalation",
    "observe_safety_signal",
    "estimate_effect_size",
    "run_interim_analysis",
    "modify_sample_size",
    "add_biomarker_stratification",
    "submit_to_fda_review",
    "request_protocol_amendment",
    "run_primary_analysis",
    "synthesize_conclusion",
    "enroll_patients",
}

REQUIRED_ACTION_ORDER = [
    "set_primary_endpoint",
    "set_sample_size",
    "set_inclusion_criteria",
    "set_dosing_schedule",
    "set_control_arm",
    "enroll_patients",
    "run_dose_escalation",
    "run_interim_analysis",
    "run_primary_analysis",
]

SYSTEM_PROMPT = """You are a clinical trial designer.\nRespond with a COMPLETE clinical action plan in JSON format.\nThe plan should contain 9 to 11 actions, one per object in an 'actions' array.\nExample: {\"actions\": [{\"action_type\": \"set_primary_endpoint\", \"parameters\": {\"endpoint\": \"overall_survival\"}}, ... ]}\nEnsure the plan ends with run_primary_analysis.\n"""
def _extract_json_candidates(text):
    candidates = []
    fenced = re.findall(r"```json?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    candidates.extend(fenced)
    candidates.append(text)
    return candidates


def _load_first_json(text):
    for candidate in _extract_json_candidates(text):
        spans = []
        obj_start, obj_end = candidate.find("{"), candidate.rfind("}")
        list_start, list_end = candidate.find("["), candidate.rfind("]")
        if obj_start != -1 and obj_end > obj_start:
            spans.append((obj_start, obj_end + 1))
        if list_start != -1 and list_end > list_start:
            spans.append((list_start, list_end + 1))
        for start, end in sorted(spans, key=lambda span: span[0]):
            try:
                return json.loads(candidate[start:end])
            except Exception:
                continue
    return None


def parse_all_actions(text):
    """Extract all valid JSON actions from model output using balanced-brace matching."""
    actions = []
    i = 0
    while i < len(text):
        if text[i] == '{':
            depth = 0
            start = i
            for j in range(i, min(i + 2000, len(text))):
                if text[j] == '{':
                    depth += 1
                elif text[j] == '}':
                    depth -= 1
                if depth == 0:
                    candidate = text[start:j+1]
                    try:
                        obj = json.loads(candidate)
                        if isinstance(obj, dict) and 'action_type' in obj:
                            actions.append(obj)
                    except json.JSONDecodeError:
                        pass
                    i = j + 1
                    break
            else:
                i += 1
        else:
            i += 1
    
    # Fallback: try ast.literal_eval
    if not actions:
        import ast
        try:
            obj = ast.literal_eval(text.strip())
            if isinstance(obj, dict) and 'action_type' in obj:
                actions.append(obj)
        except Exception:
            pass
    return actions

def parse_action_plan(text):
    """Extract and normalise the full action plan from text."""
    # Check if it's the new wrapped format {"actions": [...]}
    try:
        data = json.loads(text)
        if isinstance(data, dict) and "actions" in data:
            raw_actions = data["actions"]
        else:
            raw_actions = parse_all_actions(text)
    except:
        raw_actions = parse_all_actions(text)
    
    if not isinstance(raw_actions, list) or not raw_actions:
        # Try looking for "actions": [...] snippet
        match = re.search(r'"actions"\s*:\s*(\[.*?\])', text, re.DOTALL)
        if match:
            try:
                raw_actions = json.loads(match.group(1))
            except:
                raw_actions = parse_all_actions(text)
        else:
            raw_actions = parse_all_actions(text)

    actions = []
    for raw in raw_actions[:PLAN_MAX_ACTIONS]:
        norm = _normalise_action(raw)
        if norm:
            actions.append(norm)
            
    return {"actions": actions, "parsed": len(actions) > 0}
def _bounded_int(value, minimum, maximum):
    try:
        value = int(float(value))
    except Exception:
        return None
    return max(minimum, min(maximum, value))


def _normalise_action(raw):
    if not isinstance(raw, dict):
        return None
    action_type = raw.get("action_type")
    if action_type not in ALL_ACTION_TYPES:
        return None

    params = raw.get("parameters", {})
    if params is None:
        params = {}
    if not isinstance(params, dict):
        return None
    params = dict(params)

    if action_type == "set_sample_size":
        raw_sample_size = params.get("sample_size", params.get("n_patients"))
        if raw_sample_size is None:
            raw_sample_size = DEFAULT_SAMPLE_SIZE
        sample_size = _bounded_int(raw_sample_size, 30, 500)
        if sample_size is None:
            return None
        params["sample_size"] = sample_size

    if action_type == "enroll_patients":
        raw_n_patients = params.get("n_patients", params.get("sample_size"))
        if raw_n_patients is None:
            raw_n_patients = DEFAULT_SAMPLE_SIZE
        n_patients = _bounded_int(raw_n_patients, 0, 500)
        if n_patients is None:
            return None
        params["n_patients"] = n_patients

    return {"action_type": action_type, "parameters": params}


def parse_action_plan(text):
    """Strict full-plan parse. Legacy sample_size/add_biomarker JSON is rejected."""
    payload = _load_first_json(str(text))
    if isinstance(payload, dict):
        raw_actions = payload.get("actions")
    elif isinstance(payload, list):
        raw_actions = payload
    else:
        return {"actions": [], "parsed": False}

    if not isinstance(raw_actions, list) or not raw_actions:
        return {"actions": [], "parsed": False}

    actions = []
    for raw in raw_actions[:PLAN_MAX_ACTIONS]:
        action = _normalise_action(raw)
        if action is None:
            return {"actions": [], "parsed": False}
        actions.append(action)
    return {"actions": actions, "parsed": True}


def plan_progress_bonus(actions):
    """Differentiate valid prefixes so GRPO does not flatten all incomplete plans."""
    matched = 0
    for action in actions:
        if matched >= len(REQUIRED_ACTION_ORDER):
            break
        if action["action_type"] == REQUIRED_ACTION_ORDER[matched]:
            matched += 1
    if matched <= 1:
        return 0.0
    return MAX_INCOMPLETE_PROGRESS_BONUS * (matched / len(REQUIRED_ACTION_ORDER))


def build_reference_plan(sample_size=260):
    return {
        "actions": [
            {"action_type": "set_primary_endpoint", "parameters": {"endpoint": "overall_survival"}},
            {"action_type": "set_sample_size", "parameters": {"sample_size": sample_size}},
            {"action_type": "set_inclusion_criteria", "parameters": {}},
            {"action_type": "set_dosing_schedule", "parameters": {}},
            {"action_type": "set_control_arm", "parameters": {}},
            {"action_type": "enroll_patients", "parameters": {"n_patients": sample_size}},
            {"action_type": "run_dose_escalation", "parameters": {}},
            {"action_type": "run_interim_analysis", "parameters": {}},
            {"action_type": "run_primary_analysis", "parameters": {}},
        ]
    }


def build_design_prompt(obs):
    """Ask for a complete plan without giving the model a copy-paste plan."""
    scenario = obs.get("scenario_description", "Clinical trial scenario")
    phase = obs.get("phase_data", {}).get("current_phase", "unknown")
    res = obs.get("resource_status", {})
    budget = float(res.get("budget_remaining", 0))
    time_days = int(res.get("time_remaining_days", 0))
    available = obs.get("available_actions", [])
    return (
        f"Scenario: {scenario}\n"
        f"Current phase: {phase}\n"
        f"Budget remaining: ${budget:,.0f}\n"
        f"Time remaining: {time_days} days\n"
        f"Currently available actions: {available}\n"
        f"Minimum action_type order for completion: {REQUIRED_ACTION_ORDER}\n"
        "For higher reward, include one useful information action when legal, such as add_biomarker_stratification or estimate_effect_size. "
        "Set sample_size and enroll_patients to the same integer between 120 and 420. "
        "Use 9 to 11 actions and end with run_primary_analysis. "
        "The environment executes exactly your listed actions. "
        "Invalid, unparsable, or incomplete plans receive low reward. "
        "Return ONLY valid JSON: {\"actions\": [action objects]}. Each action object needs action_type and parameters. "
        "Template: {\"actions\":[{\"action_type\":\"set_primary_endpoint\",\"parameters\":{\"endpoint\":\"overall_survival\"}}, {\"action_type\":\"set_sample_size\",\"parameters\":{\"sample_size\":240}}, ...]}"
    )

def rollout_plan_reward(model_response, seed):
    """Execute the model's plan exactly; add noise to floor rewards for GRPO variance."""
    import random
    noise = random.uniform(0, 0.05)
    try:
        parsed = parse_action_plan(model_response)
        if not parsed["parsed"]:
            return PLAN_PARSE_FAILURE_REWARD + noise

        env_reset(seed=seed)
        total_reward = 0.0
        done = False

        for action in parsed["actions"]:
            step_result = env_step(
                action["action_type"],
                action.get("parameters", {}),
                justification="model full-plan rollout",
                confidence=0.7,
            )
            info = step_result.get("info", {})
            if not info.get("action_valid", True):
                # Still reward the progress made so far before the invalid action
                return total_reward + INVALID_SEQUENCE_REWARD + noise

            total_reward += extract_total_reward(step_result)
            done = bool(step_result.get("done", False))
            if done:
                return total_reward

        if not done:
            return total_reward + plan_progress_bonus(parsed["actions"]) + INCOMPLETE_PLAN_PENALTY + noise
        return total_reward
    except Exception as e:
        print(f"Episode error: {e}")
        return PLAN_PARSE_FAILURE_REWARD + noise
def random_action_plan(max_actions=PLAN_MAX_ACTIONS):
    actions = []
    for _ in range(random.randint(1, max_actions)):
        action_type = random.choice(sorted(ALL_ACTION_TYPES))
        params = {}
        if action_type == "set_primary_endpoint":
            params = {"endpoint": random.choice(["overall_survival", "pfs", "response_rate"])}
        elif action_type == "set_sample_size":
            params = {"sample_size": random.randint(30, 500)}
        elif action_type == "enroll_patients":
            params = {"n_patients": random.randint(0, 500)}
        actions.append({"action_type": action_type, "parameters": params})
    return {"actions": actions}


def summarize_plan(text):
    parsed = parse_action_plan(text)
    if not parsed["parsed"]:
        return "unparsed"
    actions = parsed["actions"]
    first = actions[0]["action_type"] if actions else "none"
    return f"steps={len(actions)}, first={first}, progress_bonus={plan_progress_bonus(actions):.2f}"


def completion_to_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        for key in ("content", "text", "generated_text", "completion"):
            value = completion.get(key)
            if isinstance(value, str) and value.strip():
                return value
        message = completion.get("message")
        if isinstance(message, dict):
            extracted = completion_to_text(message)
            if extracted.strip():
                return extracted
        messages = completion.get("messages")
        if isinstance(messages, list):
            extracted = completion_to_text(messages)
            if extracted.strip():
                return extracted
    if isinstance(completion, (list, tuple)):
        for item in reversed(completion):
            extracted = completion_to_text(item)
            if extracted.strip():
                return extracted
    try:
        return json.dumps(completion)
    except Exception:
        return str(completion)


_reward_step = [0]

def reward_func(completions, **kwargs):
    _reward_step[0] += 1
    group_seed = SEED + _reward_step[0] * 137

    rewards = []
    summaries = []
    for completion in completions:
        text = completion_to_text(completion)
        rewards.append(rollout_plan_reward(text, seed=group_seed))
        summaries.append(summarize_plan(text))

    if _reward_step[0] <= 5 or _reward_step[0] % 10 == 0:
        print(
            f"reward_step={_reward_step[0]} "
            f"min={min(rewards):+.2f} max={max(rewards):+.2f} "
            f"unique_rewards={len(set(round(r, 3) for r in rewards))} "
            f"plans={summaries[:3]}"
        )
    return rewards


if not DRY_RUN:
    print("Testing full-plan reward variance on seed 42...")
    test_cases = [
        (json.dumps(build_reference_plan(260)), "complete valid plan"),
        (json.dumps({"actions": [
            {"action_type": "set_primary_endpoint", "parameters": {"endpoint": "overall_survival"}},
            {"action_type": "set_sample_size", "parameters": {"sample_size": 260}},
            {"action_type": "set_inclusion_criteria", "parameters": {}},
            {"action_type": "set_dosing_schedule", "parameters": {}},
            {"action_type": "set_control_arm", "parameters": {}},
            {"action_type": "enroll_patients", "parameters": {"n_patients": 260}},
            {"action_type": "run_dose_escalation", "parameters": {}},
            {"action_type": "estimate_effect_size", "parameters": {}},
            {"action_type": "run_interim_analysis", "parameters": {}},
            {"action_type": "run_primary_analysis", "parameters": {}},
        ]}), "complete informative plan"),
        (json.dumps({"actions": [{"action_type": "set_primary_endpoint", "parameters": {"endpoint": "overall_survival"}}]}), "short incomplete plan"),
        (json.dumps({"actions": [
            {"action_type": "set_primary_endpoint", "parameters": {"endpoint": "overall_survival"}},
            {"action_type": "set_sample_size", "parameters": {"sample_size": 260}},
            {"action_type": "set_inclusion_criteria", "parameters": {}},
            {"action_type": "set_dosing_schedule", "parameters": {}},
            {"action_type": "set_control_arm", "parameters": {}},
        ]}), "longer valid prefix"),
        (json.dumps({"actions": [{"action_type": "synthesize_conclusion", "parameters": {}}]}), "invalid first action"),
        ('{"sample_size": 60, "add_biomarker": false}', "legacy two-field JSON"),
        ("enroll as many as possible", "garbage text"),
    ]
    test_rewards = []
    for text, label in test_cases:
        r = rollout_plan_reward(text, seed=42)
        test_rewards.append(r)
        print(f"  {label:25s} -> reward = {r:+.2f} | {summarize_plan(text)}")

    import numpy as np
    std = float(np.std(test_rewards))
    print(f"\nReward std: {std:.2f} (target > 1.0 for GRPO)")


## 5. Prepare Training Dataset

Prompts include scenario + resource state and require a complete action plan. The model must learn both clinical ordering and design parameters from reward feedback.


In [ ]:
if not DRY_RUN:
    from datasets import Dataset

    print(f"Generating {EPISODES} training prompts from environment...")
    train_prompts = []
    for i in range(EPISODES):
        prompt_seed = SEED + i * 7
        try:
            obs = env_reset(seed=prompt_seed)
            user_msg = build_design_prompt(obs)
            train_prompts.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
            })
        except Exception as e:
            print(f"  Prompt {i} failed: {e}")

    train_dataset = Dataset.from_list(train_prompts)
    train_dataset = train_dataset.shuffle(seed=SEED)
    print(f"Dataset ready: {len(train_dataset)} prompts")
else:
    print("DRY_RUN=True - skipping dataset creation.")

## 6. Configure and Run GRPO Training

GRPO generates `num_generations` completions per prompt, scores them with the reward function, and updates the policy to favor higher-reward completions.

In [ ]:
if not DRY_RUN:
    import torch
    from trl import GRPOConfig, GRPOTrainer

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16

    training_args = GRPOConfig(
        output_dir="checkpoints/grpo_clinical_trial",
        num_generations=NUM_GENERATIONS,
        max_completion_length=MAX_COMPLETION_LENGTH,
        temperature=GENERATION_TEMPERATURE,
        learning_rate=1e-5,
        num_train_epochs=1,
        per_device_train_batch_size=NUM_GENERATIONS,
        gradient_accumulation_steps=1,
        max_steps=EPISODES,
        warmup_steps=max(4, EPISODES // 10),
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=1,
        save_steps=max(1, EPISODES // 4),
        report_to="none",
        bf16=use_bf16,
        fp16=use_fp16,
    )

    trainer = GRPOTrainer(
        model=model,
        args=training_args,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        reward_funcs=[reward_func],
    )

    print(f"bf16={use_bf16}, fp16={use_fp16}")
    print(f"GRPO: steps={EPISODES}, generations/step={NUM_GENERATIONS}, max_completion={MAX_COMPLETION_LENGTH}")
else:
    print("DRY_RUN=True - skipping trainer setup.")




In [ ]:
if not DRY_RUN:
    print("Starting GRPO training...")
    print("=" * 60)
    TRAIN_START_TIME = datetime.now(timezone.utc)
    trainer.train()
    TRAIN_END_TIME = datetime.now(timezone.utc)

    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    reward_rows = []
    for idx, log in enumerate(trainer.state.log_history, start=1):
        if "reward" not in log:
            continue
        reward_rows.append({
            "step": int(log.get("step", idx)),
            "reward": float(log.get("reward", 0.0)),
            "reward_std": float(log.get("reward_std", 0.0)),
            "loss": float(log.get("loss", log.get("training_loss", 0.0)) or 0.0),
            "epoch": float(log.get("epoch", 0.0) or 0.0),
        })

    if reward_rows:
        csv_path = os.path.join(ARTIFACT_DIR, "reward_log.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["step", "reward", "reward_std", "loss", "epoch"])
            writer.writeheader()
            writer.writerows(reward_rows)

        rewards = [row["reward"] for row in reward_rows]
        import numpy as np
        slope = float(np.polyfit(range(len(rewards)), rewards, 1)[0]) if len(rewards) > 1 else 0.0

        summary = {
            "model_size": MODEL_SIZE,
            "episodes": EPISODES,
            "num_generations": NUM_GENERATIONS,
            "seed": SEED,
            "mean_reward": float(sum(rewards) / len(rewards)),
            "final_reward": float(rewards[-1]),
            "max_reward": float(max(rewards)),
            "min_reward": float(min(rewards)),
            "trend_slope": slope,
            "steps_logged": len(reward_rows),
            "runtime_seconds": float((TRAIN_END_TIME - TRAIN_START_TIME).total_seconds()),
            "artifact_dir": ARTIFACT_DIR,
            "completed_at": TRAIN_END_TIME.isoformat(),
        }
        summary_path = os.path.join(ARTIFACT_DIR, "training_summary.json")
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)

        print(f"Saved training artifacts: {summary_path}, {csv_path}")
        print(f"Reward trend slope: {slope:+.4f}")
    else:
        print("No reward rows captured; skipped training artifact export.")

    print("=" * 60)
    print("Training complete!")
else:
    print("DRY_RUN=True - skipping training.")

## 7. Evaluate Trained Model

Compare the trained model against a random baseline to demonstrate learning.

In [ ]:
if not DRY_RUN:
    import torch

    def generate_plan(prompt_messages, temperature=0.1):
        inputs = tokenizer.apply_chat_template(
            prompt_messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        if getattr(model, "generation_config", None) is not None:
            model.generation_config.max_length = None
        with torch.inference_mode():
            outputs = model.generate(
                inputs,
                max_new_tokens=EVAL_MAX_NEW_TOKENS,
                max_length=None,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    model.eval()
    eval_base_seed = 9000

    print(f"Evaluating RANDOM full-plan baseline ({EVAL_EPISODES} episodes)...")
    random_results = []
    for i in range(EVAL_EPISODES):
        random.seed(SEED + 5000 + i)
        fake_json = json.dumps(random_action_plan())
        r = rollout_plan_reward(fake_json, seed=eval_base_seed + i)
        random_results.append(r)
        print(f"  random  {i+1}: {summarize_plan(fake_json):28s} -> reward={r:+.2f}")

    print(f"\nEvaluating TRAINED model ({EVAL_EPISODES} episodes)...")
    trained_results = []
    for i in range(EVAL_EPISODES):
        obs = env_reset(seed=eval_base_seed + i)
        prompt_msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_design_prompt(obs)},
        ]
        response = generate_plan(prompt_msgs, temperature=0.1)
        r = rollout_plan_reward(response, seed=eval_base_seed + i)
        trained_results.append(r)
        print(f"  trained {i+1}: {summarize_plan(response):28s} -> reward={r:+.2f}")

    random_avg = sum(random_results) / len(random_results)
    trained_avg = sum(trained_results) / len(trained_results)

    eval_report = {
        "episodes": EVAL_EPISODES,
        "random_rewards": random_results,
        "trained_rewards": trained_results,
        "random_avg": random_avg,
        "trained_avg": trained_avg,
        "improvement": trained_avg - random_avg,
        "completed_at": datetime.now(timezone.utc).isoformat(),
    }
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    with open(os.path.join(ARTIFACT_DIR, "eval_report.json"), "w") as f:
        json.dump(eval_report, f, indent=2)

    print(f"\n{'='*54}")
    print(f"{'Metric':<28} {'Random':>11} {'Trained':>11}")
    print(f"{'='*54}")
    print(f"{'Avg Reward':<28} {random_avg:>11.2f} {trained_avg:>11.2f}")
    print(f"{'Improvement':<28} {'':>11} {trained_avg - random_avg:>+11.2f}")
    print(f"{'='*54}")
else:
    print("DRY_RUN=True - skipping evaluation.")


## 8. Save Checkpoint to HuggingFace Hub

Upload the trained LoRA adapter to HuggingFace for deployment and demo.

In [ ]:
if not DRY_RUN:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get('HF_TOKEN'))
        print("Logged in to HuggingFace via Colab secret.")
    except Exception:
        print("Set HF_TOKEN in Colab Secrets (key icon in sidebar) or login manually:")
        from huggingface_hub import notebook_login
        notebook_login()
else:
    print("DRY_RUN=True â€” skipping HF login.")

In [ ]:
if not DRY_RUN:
    from huggingface_hub import HfApi

    REPO_ID = "Roopalgn/clinical-trial-designer-grpo"
    api = HfApi()
    api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
    print(f"Model repo ready: https://huggingface.co/{REPO_ID}")

    model.save_pretrained("checkpoints/grpo_clinical_trial/final")
    tokenizer.save_pretrained("checkpoints/grpo_clinical_trial/final")

    model.push_to_hub(REPO_ID, commit_message=f"GRPO {MODEL_SIZE} trained on Colab")
    tokenizer.push_to_hub(REPO_ID, commit_message=f"GRPO {MODEL_SIZE} trained on Colab")

    print(f"Model uploaded to: https://huggingface.co/{REPO_ID}")
else:
    print("DRY_RUN=True â€” skipping checkpoint save.")

## 9. Generate Reward Curve Plot

Quick visualization of training progress for the pitch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not DRY_RUN and 'trainer' in dir():
    if hasattr(trainer, 'state') and trainer.state.log_history:
        rewards = [log.get('reward', None) for log in trainer.state.log_history if 'reward' in log]
        if rewards:
            episodes_list = range(1, len(rewards) + 1)
            window = min(20, len(rewards) // 3 + 1)
            rolling_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
            z = np.polyfit(range(len(rewards)), rewards, 1)
            trend = np.poly1d(z)

            fig, ax = plt.subplots(figsize=(12, 6))
            ax.scatter(episodes_list, rewards, alpha=0.3, color='#4a90d9', s=20, label='Per-episode')
            ax.plot(range(window, len(rewards) + 1), rolling_avg, color='#e63946',
                    linewidth=2, label=f'Rolling avg (w={window})')
            ax.plot(episodes_list, trend(range(len(rewards))), '--', color='#2d6a4f',
                    linewidth=1.5, label=f'Trend (slope={z[0]:.3f})')
            ax.set_xlabel('Episode', fontsize=12)
            ax.set_ylabel('Total Reward', fontsize=12)
            ax.set_title(f'Training Reward Curve â€” {MODEL_SIZE} on Colab', fontsize=14)
            ax.legend(loc='upper left')
            ax.grid(True, alpha=0.3)
            ax.annotate(
                f'Best: {max(rewards):.1f}\nFinal avg: {np.mean(rewards[-20:]):.1f}\nSlope: {z[0]:.3f}',
                xy=(0.98, 0.02), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
            )
            os.makedirs("results", exist_ok=True)
            os.makedirs(ARTIFACT_DIR, exist_ok=True)
            plt.tight_layout()
            plt.savefig('results/reward_curve.png', dpi=150)
            plt.savefig(os.path.join(ARTIFACT_DIR, 'reward_curve.png'), dpi=150)
            plt.show()
            print(f"Saved to results/reward_curve.png and {os.path.join(ARTIFACT_DIR, 'reward_curve.png')}")
        else:
            print("No reward data in training logs.")
    else:
        print("No training logs available.")
else:
    print("DRY_RUN or no trainer â€” skipping plot.")

## Summary

This notebook demonstrates:
1. **Environment Innovation** â€” a clinical trial simulator with hidden ground truth, multi-layer verification, and 5-tier curriculum
2. **Reward Design** â€” 6 per-step + 2 terminal components with potential-based shaping
3. **Training** â€” GRPO with Unsloth/TRL producing measurable improvement
4. **Proof of Learning** â€” reward curves trending upward, trained model beating random baseline

For full documentation, see the [GitHub repository](https://github.com/Roopalgn/openenv-clinical-trial).